# 06 — Fine-tune the forecasting model

Fine-tune a text-only Qwen model with 4-bit QLoRA on the panel examples produced by notebook 05. This notebook never uses the test split. It stops before loading a base model unless the panel gate passes and a CUDA GPU is available.

**Inputs:** `configs/model.yaml`, `configs/eval.yaml`, `data/manifests/panel_dataset_card.json`, `data/processed/panel_train.jsonl`, and `data/processed/panel_validation.jsonl`.

**Outputs:** an immutable model-and-timestamp run folder under `models/adapters/`, matching reports under `reports/finetune_runs/`, an immutable run manifest under `data/manifests/finetune_runs/`, and `data/manifests/finetune_run_manifest.json` as the latest-run pointer.

## 1. Mount the Colab project

This cell connects Google Drive and points every later path at the JobAI project. Run this notebook in a fresh GPU-enabled Colab runtime.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.chdir("/content/drive/MyDrive/JobAI")
os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HOME"] = "/content/drive/MyDrive/JobAI/.cache/huggingface"
os.environ["HF_DATASETS_CACHE"] = "/content/drive/MyDrive/JobAI/.cache/huggingface/datasets"
os.environ["HF_HUB_CACHE"] = "/content/drive/MyDrive/JobAI/.cache/huggingface/hub"
os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)

## 2. Install the pinned training stack

This installs only the fine-tuning libraries from `requirements-train-colab.txt`. Colab's managed PyTorch, CUDA, pandas, NumPy, and notebook packages are deliberately left unchanged. Restart the runtime only if Colab explicitly asks you to, then continue from Section 3.

In [ ]:
%pip install --quiet -r requirements-train-colab.txt

## 3. Gate, files, and GPU preflight

This code loads the project settings and panel dataset card, confirms the fine-tuning gate passed, checks that training and validation files exist, and records the available GPU. The base model is not loaded in this section.

In [ ]:
import hashlib, importlib.metadata, json, os, platform, re, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import yaml

def _find_repo():
    configured = os.environ.get("JOBAI_REPO")
    if configured:
        return Path(configured).resolve()
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "model.yaml").is_file():
            return candidate
    return current

REPO = _find_repo()
PRO = REPO / "data" / "processed"
MAN = REPO / "data" / "manifests"
REPORTS = REPO / "reports"
MODELS = REPO / "models"
for directory in (MAN, REPORTS, MODELS):
    directory.mkdir(parents=True, exist_ok=True)
MODEL_CFG = yaml.safe_load((REPO / "configs" / "model.yaml").read_text())
EVAL_CFG = yaml.safe_load((REPO / "configs" / "eval.yaml").read_text())
CARD_PATH = MAN / "panel_dataset_card.json"
TRAIN_PATH = PRO / "panel_train.jsonl"
VAL_PATH = PRO / "panel_validation.jsonl"
assert CARD_PATH.is_file(), "Run notebook 05 first: panel dataset card is missing"
panel_card = json.loads(CARD_PATH.read_text())
assert panel_card.get("gate", {}).get("finetuning_ready") is True, "Fine-tuning gate failed; notebook 06 stops before model loading"
assert TRAIN_PATH.is_file() and VAL_PATH.is_file(), "Training or validation split is missing; rerun notebook 05"
assert torch.cuda.is_available(), "A CUDA GPU is required. In Colab choose Runtime > Change runtime type > GPU"
gpu = torch.cuda.get_device_properties(0)
GPU_NAME = gpu.name
GPU_VRAM_GB = gpu.total_memory / 1024**3
BF16 = bool(torch.cuda.is_bf16_supported())
SEED = int(MODEL_CFG["training"]["seed"])
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("panel gate: PASS")
print("training examples:", panel_card["gate"]["train_examples"])
print("GPU:", GPU_NAME, f"({GPU_VRAM_GB:.1f} GB), bf16={BF16}")

## 4. Select the configured model

Load the exact candidate named by `candidate_to_run` in `model.yaml` and stop if the GPU does not meet its configured memory requirement. The notebook never silently substitutes a fallback model.

In [ ]:
candidate_by_id = {candidate["id"]: candidate for candidate in MODEL_CFG["candidates"]}
requested_model_id = MODEL_CFG.get("candidate_to_run", "Qwen/Qwen3-4B")
assert requested_model_id in candidate_by_id, f"Unknown candidate_to_run: {requested_model_id}"
MODEL_CHOICE = candidate_by_id[requested_model_id]
MINIMUM_VRAM_GB = float(MODEL_CHOICE.get("minimum_vram_gb", 14))
MODEL_ARCHITECTURE = MODEL_CHOICE.get("architecture", "multimodal_text_only" if "Qwen3.5" in requested_model_id else "causal_lm")
FREEZE_VISION = bool(MODEL_CHOICE.get("freeze_vision", MODEL_ARCHITECTURE == "multimodal_text_only"))
assert GPU_VRAM_GB >= MINIMUM_VRAM_GB, (
    f"{requested_model_id} requires at least {MINIMUM_VRAM_GB:g} GB by project policy; "
    f"this GPU has {GPU_VRAM_GB:.1f} GB. Select a larger Colab GPU or explicitly change candidate_to_run."
)
assert MODEL_CHOICE["training_mode"] == "qlora_4bit"
MODEL_ID = MODEL_CHOICE["id"]
TRAIN_CFG = MODEL_CFG["training"]
LORA_CFG = MODEL_CFG["lora"]
FEATURE_SET = TRAIN_CFG.get("feature_set", "legacy_v1")
TRAINING_HORIZONS = sorted({int(h) for h in TRAIN_CFG.get("training_horizons", [1, 2, 4])})
assert TRAINING_HORIZONS and set(TRAINING_HORIZONS).issubset({1, 2, 4})
EXPERIMENT_SLUG = f"{FEATURE_SET}__h{'-'.join(map(str, TRAINING_HORIZONS))}"
MODEL_SLUG = re.sub(r"[^a-z0-9]+", "-", MODEL_ID.lower()).strip("-")
RUN_SIGNATURE = f"{MODEL_ID}|{EXPERIMENT_SLUG}"
previous_run_complete = bool(globals().get("OUTPUT_DIR") and (Path(OUTPUT_DIR) / "final_adapter").exists())
if globals().get("FINETUNE_RUN_SIGNATURE") != RUN_SIGNATURE or previous_run_complete:
    FINETUNE_RUN_ID = f"{MODEL_SLUG}__{EXPERIMENT_SLUG}__{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}"
    FINETUNE_RUN_SIGNATURE = RUN_SIGNATURE
OUTPUT_ROOT = REPO / TRAIN_CFG.get("output_root", "models/adapters")
OUTPUT_DIR = OUTPUT_ROOT / FINETUNE_RUN_ID
RUN_REPORTS = REPORTS / "finetune_runs" / FINETUNE_RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_REPORTS.mkdir(parents=True, exist_ok=True)
assert not (OUTPUT_DIR / "final_adapter").exists(), f"Completed run already exists: {OUTPUT_DIR}"
print("run id        :", FINETUNE_RUN_ID)
print("selected model:", MODEL_ID)
print("training mode :", MODEL_CHOICE["training_mode"])
print("feature set   :", FEATURE_SET)
print("horizons      :", TRAINING_HORIZONS)
print("output folder :", OUTPUT_DIR)

## 5. Build versioned prompt-completion records

Filter to the configured forecast horizons and construct either the reproducible legacy prompt or the enhanced prompt. Enhanced prompts add only features calculated by notebook 05 at or before the forecast origin. The feature set and horizon selection are part of the cache key and immutable run ID.


In [ ]:
from datasets import Dataset, load_from_disk

SYSTEM_TEXT = ("You forecast Finnish registered job vacancies. "
               "Return exactly one JSON object with one numeric field named target_scaled_change.")

def format_number(value):
    return format(float(value), ".6g")

FEATURE_ALIASES = [
    ("quarter_of_year", "quarter"), ("qoq_change_scaled", "qoq"),
    ("yoy_change_scaled", "yoy"), ("recent_mean_4_scaled", "mean4"),
    ("window_mean_scaled", "mean_window"), ("recent_slope_4_scaled", "slope4"),
    ("window_slope_scaled", "slope_window"), ("recent_std_4_scaled", "std4"),
    ("window_std_scaled", "std_window"), ("zero_fraction_window", "zero_fraction"),
]

def compact_feature_text(engineered):
    return "; ".join(f"{alias}={format_number(engineered[key])}" for key, alias in FEATURE_ALIASES)

def make_record(row):
    history = json.loads(row["input_values_json"])
    dimensions = json.loads(row["dimensions_json"])
    feature_text = ""
    if FEATURE_SET == "enhanced_v1":
        assert row.get("engineered_features_json"), "Run updated notebook 05: enhanced features are missing"
        engineered = json.loads(row["engineered_features_json"])
        feature_text = f"Origin-safe features: {compact_feature_text(engineered)}\n"
    elif FEATURE_SET != "legacy_v1":
        raise ValueError(f"Unknown feature_set: {FEATURE_SET}")
    history_description = ("Eight quarterly vacancy values" if FEATURE_SET == "legacy_v1" and len(history) == 8
                           else f"{len(history)} quarterly vacancy values")
    user_text = (
        f"Series family: {row['series_family']}\n"
        f"Table: {row['table_id']}\n"
        f"Dimensions: {json.dumps(dimensions, ensure_ascii=False, sort_keys=True)}\n"
        f"History start: {row['window_start_quarter']}\n"
        f"{history_description}, oldest to newest: {[format_number(v) for v in history]}\n"
        f"{feature_text}"
        f"Forecast origin: {row['origin_quarter']}\n"
        f"Forecast horizon: {int(row['horizon_q'])} quarter(s)\n"
        f"Target quarter: {row['target_quarter']}\n"
        f"Scale: {format_number(row['scale'])}\n"
        "Predict target_scaled_change = (target vacancy value - latest history value) / scale."
    )
    completion = json.dumps({"target_scaled_change": round(float(row["target_scaled_change"]), 6)})
    return {
        "prompt": [{"role": "system", "content": SYSTEM_TEXT}, {"role": "user", "content": user_text}],
        "completion": [{"role": "assistant", "content": completion}],
        "example_id": row["example_id"],
        "chat_template_kwargs": {"enable_thinking": False},
    }

def deterministic_budget_sample(frame, limit, seed):
    if not limit or len(frame) <= int(limit):
        return frame.copy().reset_index(drop=True)
    ranked = frame.copy()
    ranked["_sample_key"] = ranked["example_id"].map(
        lambda value: hashlib.sha256(f"{seed}|{value}".encode()).hexdigest()
    )
    return ranked.sort_values("_sample_key").head(int(limit)).drop(columns="_sample_key").reset_index(drop=True)

TRAIN_TOTAL_AVAILABLE = int(panel_card["files"]["train"]["rows"])
VAL_TOTAL_AVAILABLE = int(panel_card["files"]["validation"]["rows"])
cache_spec = {
    "version": "prompt_v3_feature_aware", "seed": SEED,
    "model_id": MODEL_ID, "feature_set": FEATURE_SET, "training_horizons": TRAINING_HORIZONS,
    "max_seq_length": int(TRAIN_CFG["max_seq_length"]),
    "train_sha256": panel_card["files"]["train"]["sha256"],
    "validation_sha256": panel_card["files"]["validation"]["sha256"],
    "max_train_examples": TRAIN_CFG.get("max_train_examples"),
    "max_validation_examples": TRAIN_CFG.get("max_validation_examples"),
}
cache_key = hashlib.sha256(json.dumps(cache_spec, sort_keys=True).encode()).hexdigest()[:16]
CACHE_DIR = REPO / "data" / "cache" / "finetune" / cache_key
TRAIN_FRAME_CACHE = CACHE_DIR / "train_sample.parquet"
VAL_FRAME_CACHE = CACHE_DIR / "validation_sample.parquet"
TRAIN_DATASET_CACHE = CACHE_DIR / "train_dataset"
VAL_DATASET_CACHE = CACHE_DIR / "validation_dataset"
TOKENIZED_TRAIN_CACHE = CACHE_DIR / "tokenized_train_dataset"
TOKENIZED_VAL_CACHE = CACHE_DIR / "tokenized_validation_dataset"
TOKENIZED_READY = CACHE_DIR / "tokenized_complete.json"
CACHE_READY = CACHE_DIR / "cache_complete.json"
if CACHE_READY.is_file():
    train_frame = pd.read_parquet(TRAIN_FRAME_CACHE)
    val_frame = pd.read_parquet(VAL_FRAME_CACHE)
    train_dataset = load_from_disk(str(TRAIN_DATASET_CACHE))
    val_dataset = load_from_disk(str(VAL_DATASET_CACHE))
    print("Using saved prompt datasets:", CACHE_DIR)
else:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    train_frame_full = pd.read_json(TRAIN_PATH, lines=True)
    val_frame_full = pd.read_json(VAL_PATH, lines=True)
    train_frame_full = train_frame_full[train_frame_full["horizon_q"].isin(TRAINING_HORIZONS)].reset_index(drop=True)
    val_frame_full = val_frame_full[val_frame_full["horizon_q"].isin(TRAINING_HORIZONS)].reset_index(drop=True)
    train_frame = deterministic_budget_sample(train_frame_full, TRAIN_CFG.get("max_train_examples"), SEED)
    val_frame = deterministic_budget_sample(val_frame_full, TRAIN_CFG.get("max_validation_examples"), SEED + 1)
    train_records = [make_record(row) for row in train_frame.to_dict(orient="records")]
    val_records = [make_record(row) for row in val_frame.to_dict(orient="records")]
    train_dataset = Dataset.from_list(train_records)
    val_dataset = Dataset.from_list(val_records)
    train_frame.to_parquet(TRAIN_FRAME_CACHE, index=False)
    val_frame.to_parquet(VAL_FRAME_CACHE, index=False)
    train_dataset.save_to_disk(str(TRAIN_DATASET_CACHE))
    val_dataset.save_to_disk(str(VAL_DATASET_CACHE))
    CACHE_READY.write_text(json.dumps(cache_spec, indent=2), encoding="utf-8")
    print("Saved reusable prompt datasets:", CACHE_DIR)
if TOKENIZED_READY.is_file():
    train_dataset = load_from_disk(str(TOKENIZED_TRAIN_CACHE))
    val_dataset = load_from_disk(str(TOKENIZED_VAL_CACHE))
    print("Using saved tokenized datasets:", CACHE_DIR)
assert set(train_frame["example_id"]).isdisjoint(set(val_frame["example_id"]))
assert set(train_frame["horizon_q"].astype(int).unique()).issubset(set(TRAINING_HORIZONS))
TRAIN_AVAILABLE = TRAIN_TOTAL_AVAILABLE if TRAINING_HORIZONS == [1, 2, 4] else None
VAL_AVAILABLE = VAL_TOTAL_AVAILABLE if TRAINING_HORIZONS == [1, 2, 4] else None
print(f"training records  : {len(train_dataset):,} selected from {TRAIN_TOTAL_AVAILABLE:,} total panel records")
print(f"validation records: {len(val_dataset):,} selected from {VAL_TOTAL_AVAILABLE:,} total panel records")
print("sample completion :", make_record(train_frame.iloc[0].to_dict())["completion"][0]["content"])


## 6. Load the four-bit base model and attach LoRA

The base weights are quantized to four bits to fit Colab memory. Only small LoRA adapter matrices are trained; the original model weights remain frozen.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoModelForMultimodalLM, AutoProcessor, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

BASE_MODEL_DIR = REPO / "models" / "base" / MODEL_ID.replace("/", "--")
BASE_MODEL_READY = BASE_MODEL_DIR / ".download_complete"
if BASE_MODEL_READY.is_file():
    print("Using saved base model:", BASE_MODEL_DIR)
else:
    BASE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading base model once to:", BASE_MODEL_DIR)
    snapshot_download(repo_id=MODEL_ID, local_dir=BASE_MODEL_DIR)
    BASE_MODEL_READY.write_text(MODEL_ID + "\n", encoding="utf-8")
MODEL_SOURCE = str(BASE_MODEL_DIR)
compute_dtype = torch.bfloat16 if BF16 else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
if MODEL_ARCHITECTURE == "multimodal_text_only":
    processor = AutoProcessor.from_pretrained(MODEL_SOURCE, local_files_only=True)
    tokenizer = processor.tokenizer
    model_class = AutoModelForMultimodalLM
else:
    processor = None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, use_fast=True, local_files_only=True)
    model_class = AutoModelForCausalLM
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = model_class.from_pretrained(
    MODEL_SOURCE,
    local_files_only=True,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=compute_dtype,
)
model.config.use_cache = False
peft_config = LoraConfig(
    r=int(LORA_CFG["r"]),
    lora_alpha=int(LORA_CFG["alpha"]),
    lora_dropout=float(LORA_CFG["dropout"]),
    target_modules=list(LORA_CFG["target_modules"]),
    bias=LORA_CFG["bias"],
    task_type=LORA_CFG["task_type"],
)
if FREEZE_VISION:
    for name, parameter in model.named_parameters():
        if any(marker in name.lower() for marker in ("visual", "vision_model", "vision_tower")):
            parameter.requires_grad = False
print("base model loaded in four-bit mode; vision frozen:", FREEZE_VISION)
# Catch prompt truncation before an expensive training run.
probe_frame = train_frame.sample(min(len(train_frame), 512), random_state=SEED)
probe_lengths = []
for row in probe_frame.to_dict(orient="records"):
    record = make_record(row)
    chat = tokenizer.apply_chat_template(record["prompt"] + record["completion"], tokenize=False, enable_thinking=False)
    probe_lengths.append(len(tokenizer(chat, add_special_tokens=False)["input_ids"]))
print("prompt token length (median/max):", int(np.median(probe_lengths)), "/", max(probe_lengths))
assert max(probe_lengths) <= int(TRAIN_CFG["max_seq_length"]), (
    f"Enhanced prompt needs {max(probe_lengths)} tokens but max_seq_length is {TRAIN_CFG['max_seq_length']}; "
    "increase max_seq_length or shorten the prompt before training."
)


## 7. Configure supervised fine-tuning

This creates the trainer from `model.yaml`. Loss is calculated on the assistant completion rather than the prompt. Validation loss is monitored during training, while forecast MAE and parse success are checked after training for the retained checkpoints.

In [ ]:
import inspect
import math
from trl import SFTConfig, SFTTrainer

sft_parameters = inspect.signature(SFTConfig).parameters
sft_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "num_train_epochs": float(TRAIN_CFG["num_train_epochs"]),
    "per_device_train_batch_size": int(TRAIN_CFG["per_device_train_batch_size"]),
    "per_device_eval_batch_size": int(TRAIN_CFG["per_device_train_batch_size"]),
    "gradient_accumulation_steps": int(TRAIN_CFG["gradient_accumulation_steps"]),
    "learning_rate": float(TRAIN_CFG["learning_rate"]),
    "lr_scheduler_type": TRAIN_CFG["lr_scheduler_type"],
    "warmup_ratio": float(TRAIN_CFG["warmup_ratio"]),
    "weight_decay": float(TRAIN_CFG["weight_decay"]),
    "logging_steps": int(TRAIN_CFG["logging_steps"]),
    "save_steps": int(TRAIN_CFG["save_steps"]),
    "eval_steps": int(TRAIN_CFG["eval_steps"]),
    "save_total_limit": int(TRAIN_CFG["save_total_limit"]),
    "save_strategy": "steps",
    "bf16": BF16,
    "fp16": not BF16,
    "gradient_checkpointing": bool(TRAIN_CFG["gradient_checkpointing"]),
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": TRAIN_CFG.get("optim", "adamw_torch"),
    "completion_only_loss": True,
    "report_to": "none",
    "seed": SEED,
}
sft_kwargs["max_length" if "max_length" in sft_parameters else "max_seq_length"] = int(TRAIN_CFG["max_seq_length"])
sft_kwargs["eval_strategy" if "eval_strategy" in sft_parameters else "evaluation_strategy"] = "steps"
if "warmup_ratio" not in sft_parameters and "warmup_steps" in sft_parameters:
    micro_batch = int(TRAIN_CFG["per_device_train_batch_size"])
    accumulation = int(TRAIN_CFG["gradient_accumulation_steps"])
    batches_per_epoch = math.ceil(len(train_dataset) / micro_batch)
    optimizer_steps = math.ceil(batches_per_epoch / accumulation) * int(math.ceil(float(TRAIN_CFG["num_train_epochs"])))
    sft_kwargs.pop("warmup_ratio")
    sft_kwargs["warmup_steps"] = max(1, math.ceil(optimizer_steps * float(TRAIN_CFG["warmup_ratio"])))
    print("Converted warmup_ratio to warmup_steps:", sft_kwargs["warmup_steps"])
unsupported = sorted(key for key in sft_kwargs if key not in sft_parameters)
for key in unsupported:
    sft_kwargs.pop(key)
if unsupported:
    print("SFTConfig options unavailable in this installed version:", unsupported)
training_args = SFTConfig(**sft_kwargs)
trainer_parameters = inspect.signature(SFTTrainer).parameters
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "peft_config": peft_config,
}
trainer_kwargs["processing_class" if "processing_class" in trainer_parameters else "tokenizer"] = tokenizer
trainer = SFTTrainer(**trainer_kwargs)
if not TOKENIZED_READY.is_file():
    for path in (TOKENIZED_TRAIN_CACHE, TOKENIZED_VAL_CACHE):
        if path.exists():
            shutil.rmtree(path)
    trainer.train_dataset.save_to_disk(str(TOKENIZED_TRAIN_CACHE))
    trainer.eval_dataset.save_to_disk(str(TOKENIZED_VAL_CACHE))
    TOKENIZED_READY.write_text(json.dumps(cache_spec, indent=2), encoding="utf-8")
    print("Saved reusable tokenized datasets:", CACHE_DIR)
if FREEZE_VISION:
    for name, parameter in trainer.model.named_parameters():
        if any(marker in name.lower() for marker in ("visual", "vision_model", "vision_tower")):
            parameter.requires_grad = False
actual_batch = int(trainer.args.per_device_train_batch_size)
actual_accumulation = int(trainer.args.gradient_accumulation_steps)
assert actual_batch == int(TRAIN_CFG["per_device_train_batch_size"]), (
    f"Trainer batch {actual_batch} differs from model.yaml batch {TRAIN_CFG['per_device_train_batch_size']}"
)
assert actual_accumulation == int(TRAIN_CFG["gradient_accumulation_steps"]), (
    f"Trainer accumulation {actual_accumulation} differs from model.yaml accumulation {TRAIN_CFG['gradient_accumulation_steps']}"
)
effective_batch = actual_batch * actual_accumulation * int(getattr(trainer.args, "world_size", 1))
train_loader = trainer.get_train_dataloader()
loader_batch = getattr(train_loader, "batch_size", None)
if loader_batch is None:
    loader_batch = getattr(getattr(train_loader, "batch_sampler", None), "batch_size", None)
print("configured micro-batch       :", TRAIN_CFG["per_device_train_batch_size"])
print("trainer micro-batch          :", actual_batch)
print("dataloader batch             :", loader_batch)
print("gradient accumulation        :", actual_accumulation)
print("effective batch (all devices):", effective_batch)
trainer.model.print_trainable_parameters()

## 8. Train and save the final adapter

This is the expensive Colab step. It fine-tunes only the LoRA parameters, saves periodic checkpoints, writes the final adapter, and stores the trainer log for later review.

In [ ]:
started_at = time.time()
train_result = trainer.train()
training_seconds = time.time() - started_at
FINAL_ADAPTER = OUTPUT_DIR / "final_adapter"
trainer.save_model(str(FINAL_ADAPTER))
tokenizer.save_pretrained(FINAL_ADAPTER)
if processor is not None:
    processor.save_pretrained(FINAL_ADAPTER)
training_log = pd.DataFrame(trainer.state.log_history)
TRAINING_LOG_PATH = RUN_REPORTS / "finetune_training_log.csv"
training_log.to_csv(TRAINING_LOG_PATH, index=False)
print(f"training time: {training_seconds / 3600:.2f} hours")
print("final adapter:", FINAL_ADAPTER)
print("training log :", TRAINING_LOG_PATH)

## 9. Validate output parsing and forecast accuracy

Generate forecasts for a fixed, balanced validation sample. The code checks that each response is valid JSON, reconstructs the vacancy forecast, and reports validation MAE. This uses no test examples.

In [ ]:
VALIDATION_PER_HORIZON = 96
validation_parts = [
    group.sample(min(len(group), VALIDATION_PER_HORIZON), random_state=SEED + int(horizon))
    for horizon, group in val_frame.groupby("horizon_q", sort=True)
]
validation_sample = pd.concat(validation_parts, ignore_index=True)

def prompt_messages(row):
    return make_record(row)["prompt"]

def parse_scaled_change(text):
    match = re.search(r"\{[^{}]*\}", text)
    if not match:
        return None
    try:
        payload = json.loads(match.group(0))
        value = float(payload["target_scaled_change"])
        return value if np.isfinite(value) else None
    except (KeyError, TypeError, ValueError, json.JSONDecodeError):
        return None

trainer.model.eval()
tokenizer.padding_side = "left"
model_size_b = float(MODEL_CHOICE.get("expected_size_b", 0))
VALIDATION_BATCH_SIZE = 1 if model_size_b >= 20 and GPU_VRAM_GB < 60 else (8 if model_size_b >= 20 else 16)
validation_rows = []
validation_records = validation_sample.to_dict(orient="records")
for start in range(0, len(validation_records), VALIDATION_BATCH_SIZE):
    batch = validation_records[start:start + VALIDATION_BATCH_SIZE]
    prompts = [tokenizer.apply_chat_template(
        prompt_messages(row), tokenize=False, add_generation_prompt=True, enable_thinking=False
    ) for row in batch]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=int(TRAIN_CFG["max_seq_length"])).to(trainer.model.device)
    with torch.inference_mode():
        generated = trainer.model.generate(**inputs, max_new_tokens=32, do_sample=False, use_cache=True)
    prompt_width = inputs["input_ids"].shape[1]
    responses = tokenizer.batch_decode(generated[:, prompt_width:], skip_special_tokens=True)
    for row, response in zip(batch, responses):
        response = response.strip()
        predicted_change = parse_scaled_change(response)
        parsed = predicted_change is not None
        predicted_value = max(0.0, float(row["last_value"]) + predicted_change * float(row["scale"])) if parsed else np.nan
        validation_rows.append({
            "example_id": row["example_id"], "horizon_q": int(row["horizon_q"]),
            "target_value": float(row["target_value"]), "predicted_value": predicted_value,
            "parsed": parsed, "abs_error": abs(float(row["target_value"]) - predicted_value) if parsed else np.nan,
            "response": response,
        })
    print(f"validated {min(start + len(batch), len(validation_records)):,}/{len(validation_records):,}", end="\r")
print()
checkpoint_validation = pd.DataFrame(validation_rows)
CHECKPOINT_REPORT_PATH = RUN_REPORTS / "finetune_checkpoint_validation.csv"
checkpoint_validation.to_csv(CHECKPOINT_REPORT_PATH, index=False)
VAL_PARSE_RATE = float(checkpoint_validation["parsed"].mean())
VAL_MAE = float(checkpoint_validation.loc[checkpoint_validation["parsed"], "abs_error"].mean())
print(f"validation parse success: {VAL_PARSE_RATE:.1%}")
print(f"validation MAE: {VAL_MAE:,.3f}")
display(checkpoint_validation.groupby("horizon_q").agg(parse_success=("parsed", "mean"), MAE=("abs_error", "mean"), n=("example_id", "size")).reset_index())

## 10. Save the reproducibility manifest

Record the exact data hashes, model choice, package versions, hardware, training settings, adapter location, runtime, and validation results. Notebook 07 uses the selected adapter and performs the untouched test comparison against Notebook four.

In [ ]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

package_names = ["torch", "transformers", "peft", "trl", "datasets", "accelerate", "bitsandbytes", "safetensors"]
package_versions = {name: importlib.metadata.version(name) for name in package_names}
run_manifest = {
    "run_id": FINETUNE_RUN_ID,
    "status": "colab_pilot_trained_validation_complete",
    "base_model": MODEL_ID,
    "base_model_commit": getattr(trainer.model.config, "_commit_hash", None),
    "training_mode": MODEL_CHOICE["training_mode"],
    "feature_set": FEATURE_SET,
    "training_horizons": TRAINING_HORIZONS,
    "adapter_path": str(FINAL_ADAPTER.relative_to(REPO)),
    "panel_dataset_card_sha256": sha256(CARD_PATH),
    "train_split_sha256": sha256(TRAIN_PATH),
    "validation_split_sha256": sha256(VAL_PATH),
    "test_data_used": False,
    "train_examples_available": TRAIN_AVAILABLE,
    "train_examples_used": len(train_dataset),
    "train_examples": len(train_dataset),
    "validation_examples_available": VAL_AVAILABLE,
    "validation_examples_used": len(val_dataset),
    "validation_examples": len(val_dataset),
    "validation_sample_examples": len(checkpoint_validation),
    "validation_MAE_mean": VAL_MAE,
    "parse_success_rate": VAL_PARSE_RATE,
    "training_seconds": training_seconds,
    "gpu": {"name": GPU_NAME, "vram_gb": GPU_VRAM_GB, "bf16": BF16},
    "platform": platform.platform(),
    "packages": package_versions,
    "model_config": MODEL_CFG,
    "trainer_metrics": train_result.metrics,
    "outputs": {
        "training_log": {"path": str(TRAINING_LOG_PATH.relative_to(REPO)), "sha256": sha256(TRAINING_LOG_PATH)},
        "validation_report": {"path": str(CHECKPOINT_REPORT_PATH.relative_to(REPO)), "sha256": sha256(CHECKPOINT_REPORT_PATH)},
    },
}
RUN_MANIFEST_DIR = MAN / "finetune_runs"
RUN_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
RUN_MANIFEST_PATH = RUN_MANIFEST_DIR / f"{FINETUNE_RUN_ID}.json"
LATEST_RUN_MANIFEST_PATH = MAN / "finetune_run_manifest.json"
manifest_text = json.dumps(run_manifest, indent=2, ensure_ascii=False)
RUN_MANIFEST_PATH.write_text(manifest_text)
LATEST_RUN_MANIFEST_PATH.write_text(manifest_text)
print("immutable run manifest:", RUN_MANIFEST_PATH)
print("latest run pointer    :", LATEST_RUN_MANIFEST_PATH)
print("Notebook 06 run complete. Notebook 07 must compare this immutable adapter on the untouched test split.")